### Aula 09 — PostgreSQL, JDBC e paralelismo de ingestão

O notebook percorre um fluxo de arquivos para PostgreSQL e compara a leitura JDBC de uma tabela grande com uma e quatro partições. Depois reduz linhas/colunas na origem e separa três decisões: paralelismo, lote de leitura e lote de escrita.

**Pré-requisitos:** ambiente PySpark com o driver `/opt/jdbc/postgresql.jar`, pacote `python-dotenv`, CSV em `/workspace/datasets/09_jdbc/pedidos.csv`, PostgreSQL acessível como `postgres:5432` e banco `spark_db`. As tabelas usadas precisam existir ou ser criadas conforme a operação e as permissões disponíveis. A senha é obtida de `POSTGRES_PASSWORD`; seu valor não deve entrar na documentação.

Os números comentados são os resultados já salvos; não há benchmark de duração entre as estratégias.

#### 1. Carregar dependências e iniciar Spark com o driver

`load_dotenv()` disponibiliza variáveis de ambiente do laboratório; o retorno `True` salvo não é um teste de autenticação no banco. A configuração `spark.jars` disponibiliza o driver JDBC à aplicação. `getOrCreate()` pode reutilizar uma sessão, por isso configurações de inicialização devem ser conferidas ao reproduzir. O aviso de porta informa tentativa de usar a Spark UI em 4041, sem medir desempenho.


In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from dotenv import load_dotenv
import os 

load_dotenv()

True

In [3]:
spark = (
    SparkSession.builder
    .appName("aula_09_jdbc")
    .config("spark.jars", "/opt/jdbc/postgresql.jar")
    .getOrCreate()
)

26/10/08 00:37:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/08 00:37:35 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


#### 2. Preparar os pedidos em Parquet

O CSV é lido com cabeçalho e inferência de tipos, gravado em modo `overwrite` e relido como Parquet. A escrita executa a preparação e substitui o destino indicado. A releitura aproveita os metadados de schema do Parquet, formando o DataFrame que será enviado ao banco. Esse estágio separa a preparação em arquivos da carga JDBC; o notebook não mede se ele torna a carga mais rápida.


In [4]:
df_pedidos = spark.read.csv("/workspace/datasets/09_jdbc/pedidos.csv", header= True, inferSchema= True)

In [5]:
df_pedidos.write.mode("overwrite").parquet("/workspace/datasets/09_jdbc/df_pedidos_parquet")

In [6]:
df_pedidos_parquet = spark.read.parquet("/workspace/datasets/09_jdbc/df_pedidos_parquet")

#### 3. Definir a conexão entre Spark e PostgreSQL

`jdbc_url` usa o hostname `postgres`, resolvido na rede Docker do laboratório. Dentro do container Spark, `localhost` se referiria ao próprio container, e não automaticamente ao banco.

O dicionário `credencias` combina usuário, senha do ambiente e classe `org.postgresql.Driver`. Ele é passado tanto à leitura quanto à escrita; construir o dicionário não abre uma carga nem comprova a conexão. O nome da variável é preservado conforme o código original.


In [7]:
jdbc_url = "jdbc:postgresql://postgres:5432/spark_db"

credencias = {
    "user": "postgres",
    "password": os.getenv("POSTGRES_PASSWORD"),
    "driver": "org.postgresql.Driver"
}

#### 4. Gravar a staging via JDBC

`write.jdbc()` é uma action de saída: executa o plano e envia os pedidos a `public.staging_pedidos`. `overwrite` pode remover e recriar uma tabela, conforme opções e suporte do banco; a existência prévia de índices, constraints ou tipos definidos em SQL não deve ser presumida depois dessa operação.

O código original conserva `properties={**credencias, "": ""}`. A chave vazia não documenta uma opção de tuning útil e seu efeito não é demonstrado pelas saídas. A célula também conserva `execution_count=None`; a leitura posterior evidencia o conteúdo encontrado na staging, sem provar isoladamente que esta escrita específica foi concluída no mesmo fluxo.


In [ ]:
df_pedidos_parquet.write.jdbc(
    url=jdbc_url,
    table= "public.staging_pedidos",
    mode= "overwrite",
    properties= {**credencias, "" : ""}
)

#### 5. Reler a staging e validar a carga encontrada

A leitura JDBC cria um DataFrame para `public.staging_pedidos`; `show(5)` consulta uma amostra e `count()` obtém o total. **Observado:** há 5.000 linhas, com `pedido_id`/`cliente_id` inteiros, `data_pedido` como `date`, strings em `status`/`canal` e `valor_total` como `double`.

Essas saídas verificam volume e tipos vistos pelo Spark, mas cinco linhas não validam todos os registros e 5.000 não comprova ausência de duplicidades. O mapeamento de tipos depende da definição real da tabela e do driver; a staging deste trecho não deve ser confundida com a tabela grande, cujo valor monetário será lido como decimal.


In [9]:
df_pedidos_postgres = spark.read.jdbc(
    url= jdbc_url,
    table= "public.staging_pedidos",
    properties= credencias
)

In [10]:
df_pedidos_postgres.show(5)

+---------+----------+-----------+---------+-----------+-----------+
|pedido_id|cliente_id|data_pedido|   status|      canal|valor_total|
+---------+----------+-----------+---------+-----------+-----------+
|    10001|       452| 2025-10-09|concluido|       site|     473.56|
|    10002|        71| 2025-08-03|concluido|        app|     303.13|
|    10003|       411| 2025-10-06|  enviado|       loja|     281.64|
|    10004|       184| 2025-09-03|concluido|       loja|     281.65|
|    10005|       290| 2025-09-15|concluido|marketplace|     893.22|
+---------+----------+-----------+---------+-----------+-----------+
only showing top 5 rows



In [11]:
df_pedidos_postgres.printSchema()

root
 |-- pedido_id: integer (nullable = true)
 |-- cliente_id: integer (nullable = true)
 |-- data_pedido: date (nullable = true)
 |-- status: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- valor_total: double (nullable = true)



In [12]:
df_pedidos_postgres.count()

5000

#### 6. Ler a tabela grande sem particionamento JDBC

`df_fato_large` lê `public.fato_pedidos_large` sem coluna, bounds ou número de partições. **Observado:** `count()` retorna 2.000.000 e `rdd.getNumPartitions()` retorna 1. O volume não fez o Spark paralelizar automaticamente a entrada JDBC em várias faixas.

Uma única partição pode limitar a ingestão, mesmo com executores disponíveis. Spark Partitions representam divisões de trabalho; tasks dependem também dos recursos do ambiente e das actions. Contar partições não mede velocidade nem número de CPUs efetivamente usadas. Sem persistência, actions distintas podem consultar novamente a origem; consistência temporal também importa quando a tabela muda entre leituras.


In [13]:
df_fato_large = spark.read.jdbc(
    url= jdbc_url,
    table= "public.fato_pedidos_large",
    properties= credencias
)

In [14]:
df_fato_large.count()

2000000

In [15]:
df_fato_large.rdd.getNumPartitions()

1

#### 7. Dividir a leitura em quatro faixas

A segunda leitura usa `pedido_id` como coluna de particionamento (`column` nesta API), referências `lowerBound=1` e `upperBound=2000000`, e `numPartitions=4`. Os bounds orientam o cálculo de intervalos de **valores da coluna**. Não são um `WHERE` que limita o conjunto retornado, nem representam quantidade de linhas por partição.

**Observado:** o total continua 2.000.000 e o DataFrame tem quatro partições. `show(1)` exibe apenas uma amostra; não exercita necessariamente todas as faixas como uma leitura completa. O paralelismo disponível depende das tasks agendadas e pode levar a conexões simultâneas ao PostgreSQL.

Uma coluna numérica ou chave primária não garante equilíbrio: a distribuição de valores, lacunas e concentração por faixa precisam ser examinadas. Mais faixas também pressionam CPU, disco, rede e conexões da origem; quatro é a configuração deste experimento, sem comprovação de ser a melhor.


In [16]:
df_fato_large2 = spark.read.jdbc(
    url= jdbc_url,
    table = "public.fato_pedidos_large",
    column= "pedido_id",
    lowerBound= 1,
    upperBound= 2000000,
    numPartitions= 4,
    properties= credencias
    
)

In [17]:
df_fato_large2.show(1)

+---------+----------+-----------+-------+-----+--------------------+
|pedido_id|cliente_id|data_pedido| status|canal|         valor_total|
+---------+----------+-----------+-------+-----+--------------------+
|        1|         1| 2024-01-01|enviado|  app|51.00000000000000...|
+---------+----------+-----------+-------+-----+--------------------+
only showing top 1 row



In [18]:
df_fato_large2.count()

2000000

In [19]:
df_fato_large2.rdd.getNumPartitions()

4

##### 7.1. Medir a distribuição antes da agregação

`spark_partition_id()` identifica a partição das linhas **antes** de `groupBy`. A agregação conta essas linhas por ID; `orderBy` organiza a apresentação e pode exigir redistribuição adicional. O resultado não descreve as partições finais dessa agregação.

**Observado nas quatro faixas:** 500.001, 499.999, 499.999 e 500.001 registros, somando 2.000.000. O equilíbrio pertence a este dataset. O [desafio 09](../desafios/desafio_09_jdbc.ipynb) altera o bound superior e registra concentração na última faixa, reforçando que bounds inadequados podem desequilibrar a leitura sem filtrar as linhas excedentes.


In [20]:
(
    df_fato_large2.withColumn("spark_partition", F.spark_partition_id())
    .groupby("spark_partition")
    .count()
    .orderBy("spark_partition")
    
).show()

+---------------+------+
|spark_partition| count|
+---------------+------+
|              0|500001|
|              1|499999|
|              2|499999|
|              3|500001|
+---------------+------+



#### 8. Filtrar linhas e selecionar colunas no PostgreSQL

O parâmetro `table` recebe uma subquery com alias `df_filtrado`, não uma nova tabela persistente. O SQL mantém somente `pedido_id`, `cliente_id`, `valor_total` e `status='concluido'`. A restrição acontece no banco antes da transferência de linhas pela conexão; o Spark ainda divide o resultado por faixas de `pedido_id`.

Os bounds continuam de 1 a 2.000.000 porque dizem respeito aos valores de IDs, mesmo que a consulta retorne menos linhas. Para orientar as faixas, deve-se considerar `MIN`/`MAX` e a distribuição da coluna sobre o conjunto consultado. O SQL selecionado reduz o que atravessa JDBC, mas o custo para o PostgreSQL executar esse filtro precisa ser avaliado à parte.


In [21]:
df_filtrado_jdbc = spark.read.jdbc(
    url= jdbc_url,
    table= """( 
        
        SELECT
            pedido_id,
            cliente_id,
            valor_total
        FROM public.fato_pedidos_large
        WHERE status = 'concluido'    
            
    ) AS df_filtrado""",
    column= "pedido_id",
    lowerBound= 1,
    upperBound= 2000000,
    numPartitions= 4,
    properties= credencias
    
)

In [22]:
# lowerBound e upperBound representam a faixa de VALORES da partitionColumn,
# não a quantidade de linhas retornadas pela consulta.

COUNT(*) responde:
"quantas linhas tenho?"

MIN/MAX da partitionColumn respondem:
"qual faixa estou dividindo?"

##### 8.1. Conferir volume, distribuição e tipos após o filtro

**Observado:** as contagens por partição são 125.000, 125.000, 124.999 e 125.001; somam as 500.000 linhas retornadas por `count()`. O schema contém apenas os três campos selecionados, com `valor_total: decimal(38,18)` nesta tabela.

As 500.000 linhas não transformam o maior `pedido_id` em 500.000. `WHERE` limita o conjunto; bounds dividem o espaço de valores para ingestão. O decimal também não deve ser confundido com o `double` visto na staging: preservar a precisão monetária exige conferir os tipos reais nos dois sistemas.


In [23]:
(
    df_filtrado_jdbc.withColumn("spark_partition", F.spark_partition_id())
    .groupby("spark_partition")
    .count()
    .orderBy("spark_partition")
    
).show()

+---------------+------+
|spark_partition| count|
+---------------+------+
|              0|125000|
|              1|125000|
|              2|124999|
|              3|125001|
+---------------+------+



In [24]:
df_filtrado_jdbc.count()

500000

In [25]:
df_filtrado_jdbc.printSchema()

root
 |-- pedido_id: integer (nullable = true)
 |-- cliente_id: integer (nullable = true)
 |-- valor_total: decimal(38,18) (nullable = true)



#### 9. `fetchsize`: lote de leitura, sem mudar as faixas

`properties={**credencias, "fetchsize": "100000"}` cria um novo dicionário com as credenciais e a configuração de lote, sem alterar o original. `fetchsize` orienta quantas linhas o driver busca por lote na leitura PostgreSQL → Spark; o comportamento efetivo depende do driver e da conexão.

**Observado:** `df_fetch.count()` retorna 500.000 e ambos os DataFrames filtrados continuam com quatro partições. Quem define essas faixas é a configuração de particionamento JDBC. Não há medição de tempo ou memória que demonstre vantagem do lote 100.000; ele não deve ser tratado como recomendação geral.


In [26]:
df_fetch = spark.read.jdbc(
    url= jdbc_url,
    properties= {**credencias, "fetchsize" : "100000"},
    table= """(
        
    SELECT
        pedido_id,
        cliente_id,
        valor_total
    FROM public.fato_pedidos_large
    WHERE status = 'concluido' 
    
    )AS df_fetch_filtrado""",
    column= "pedido_id",
    lowerBound= 1,
    upperBound= 2000000,
    numPartitions= 4
)

In [27]:
df_fetch.count()

500000

In [28]:
df_fetch.rdd.getNumPartitions()

4

In [29]:
df_filtrado_jdbc.rdd.getNumPartitions()

4

#### 10. `batchsize`: agrupar a escrita no banco

A action envia `df_filtrado_jdbc` para `public.pedidos_concluidos_batch` com `mode="append"` e `batchsize="5000"`. Essa opção controla o lote de linhas na direção Spark → PostgreSQL, enquanto `fetchsize` trata da direção inversa. Nenhuma delas cria novas faixas de leitura.

`append` acrescenta registros e não implementa deduplicação ou upsert. Reexecutar uma carga pode duplicar dados se o banco permitir; constraints existentes podem rejeitar registros repetidos. Esta aula não salva uma contagem da tabela após a escrita, portanto o volume efetivamente encontrado no destino não foi validado aqui. Também não há comparação de duração para provar ganho do lote.


In [31]:
df_filtrado_jdbc.write.jdbc(
    url= jdbc_url,
    properties = {**credencias, "batchsize" : "5000"},
    mode= "append",
    table = "public.pedidos_concluidos_batch"
)

#### 11. Encerrar a aplicação

`spark.stop()` encerra o contexto usado pelo laboratório. Os contadores de execução preservados não são uma cronologia estritamente crescente: a análise didática acompanha a ordem das células e distingue os outputs existentes de uma reprodução integral. Nenhuma conexão ou carga adicional é necessária para ler este material.


In [19]:
spark.stop()

#### Resumo dos aprendizados

| Decisão | Evidência deste notebook |
| --- | --- |
| Paralelismo JDBC | Leitura simples: 1 partição; por faixas de IDs: 4 partições, ambas com 2.000.000 de linhas. |
| Redução na origem | Subquery de concluídos: 500.000 linhas e 3 colunas. |
| `fetchsize` | Lote 100.000 configurado; continuam 4 partições. |
| `batchsize` | Lote 5.000 configurado na escrita `append`; destino sem contagem salva após a carga. |

Bounds não filtram o conjunto, contagem não é faixa de IDs, e faixas iguais de valores não garantem iguais volumes de linhas. A escolha de paralelismo deve respeitar também a capacidade do PostgreSQL. As saídas demonstram volume, schema e distribuição, sem comprovar aceleração.

**Lacuna para reprodução:** o SQL de apoio presente em `notebooks/CREATE TABLE public.sql` contém consultas e DDLs auxiliares, mas não a criação/população da `fato_pedidos_large` nesta cópia. Os 2 milhões são comprovados pelos outputs salvos; regenerar essa origem requer o script ausente ou uma tabela equivalente preparada à parte.

**Para continuar:** [material JDBC](../docs/09_jdbc_postgresql.md), [desafio 09](../desafios/desafio_09_jdbc.ipynb) e [aula 10 — Performance](aula_10_performance.ipynb).
